# Adult Income：收入二元分類教學範例

1. 讀取與檢視 Adult Income 資料  
2. 建立目標欄位 `income_binary`  
3. 觀察目標欄位與重要特徵的關係  
4. 使用 `pd.get_dummies()` 處理類別欄位  
5. 使用 `train_test_split` 切分訓練資料與測試資料  
6. 使用 `StandardScaler` 做標準化  
7. 使用 `LogisticRegression` 搭配 `GridSearchCV` 建立分類模型  
8. 使用 `F1-score` 與混淆矩陣評估模型  
9. 儲存 CV 後的模型，並模擬使用者填寫資料後進行預測

| 欄位名稱 | 中文說明 | 原始型態 | 資料角色 | 內容說明 | 缺漏值 / 特殊缺漏標記 | 建模時的處理建議 |
|---|---|---|---|---|---:|---|
| `age` | 年齡 | `int64` | 數值特徵 | 個人的年齡 | 0 | 可直接作為模型特徵；若使用 Logistic Regression、KNN、SVM 等模型，可考慮標準化 |
| `workclass` | 工作類型 | `object` | 類別特徵 | 例如 Private、Self-emp-not-inc、Local-gov 等 | `?` 有 2,799 筆 | 需要處理 `?`，可視為缺漏值後用眾數填補，再做 one-hot encoding |
| `fnlwgt` | 最終樣本權重 | `int64` | 數值特徵 | census final weight，表示該筆樣本在母體中的代表權重 | 0 | 入門教學可先保留或移除；若保留，建議標準化 |
| `education` | 教育程度 | `object` | 類別特徵 | 例如 HS-grad、Bachelors、Some-college、Masters 等 | 0 | 可做 one-hot encoding；但與 `educational-num` 資訊高度重疊 |
| `educational-num` | 教育程度數值編碼 | `int64` | 數值 / 序位特徵 | 教育程度的數值化版本，數值越高通常代表教育程度越高 | 0 | 可直接作為模型特徵；若已使用此欄位，可考慮不使用 `education` 以降低重複性 |
| `marital-status` | 婚姻狀態 | `object` | 類別特徵 | 例如 Married-civ-spouse、Never-married、Divorced 等 | 0 | 需要做 one-hot encoding |
| `occupation` | 職業 | `object` | 類別特徵 | 例如 Prof-specialty、Craft-repair、Exec-managerial、Sales 等 | `?` 有 2,809 筆 | 需要處理 `?`，可視為缺漏值後用眾數填補，再做 one-hot encoding |
| `relationship` | 家庭關係 | `object` | 類別特徵 | 例如 Husband、Not-in-family、Own-child、Unmarried 等 | 0 | 需要做 one-hot encoding |
| `race` | 種族 | `object` | 類別特徵 | 例如 White、Black、Asian-Pac-Islander 等 | 0 | 可做 one-hot encoding；教學時可提醒這是敏感屬性，使用時需注意公平性議題 |
| `gender` | 性別 | `object` | 類別特徵 | Male、Female | 0 | 可做 one-hot encoding；也是敏感屬性，使用時需注意公平性議題 |
| `capital-gain` | 資本利得 | `int64` | 數值特徵 | 投資或資產交易所產生的資本利得 | 0 | 數值分布可能偏態明顯；可直接使用，必要時可做標準化 |
| `capital-loss` | 資本損失 | `int64` | 數值特徵 | 投資或資產交易所產生的資本損失 | 0 | 數值分布可能偏態明顯；可直接使用，必要時可做標準化 |
| `hours-per-week` | 每週工作時數 | `int64` | 數值特徵 | 每週工作幾小時 | 0 | 可直接作為模型特徵；若使用尺度敏感模型，可考慮標準化 |
| `native-country` | 出生國家 / 原生國家 | `object` | 類別特徵 | 例如 United-States、Mexico、Philippines、Germany 等 | `?` 有 857 筆 | 需要處理 `?`，可視為缺漏值後用眾數填補，再做 one-hot encoding |
| `income` | 年收入類別 | `object` | 目標欄位 | `<=50K`、`>50K` | 0 | 二元分類目標欄位，通常轉成 0/1 |

In [1]:
# 忽略警告訊息
import warnings
warnings.filterwarnings("ignore")

# 匯入必要的套件
import matplotlib.pyplot as plt

# 設定字型，確保中文能顯示微軟正黑體
plt.rcParams["font.family"] = "Microsoft JhengHei"

# 設定數學字型，確保負號能正常顯示
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["mathtext.fontset"] = "dejavusans"

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

In [3]:
# 讀取資料
df = pd.read_csv('./adult_income.csv')

In [4]:
# 檢視資料
display(df.info())
display(df.describe())
display(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              48842 non-null  int64
 1   workclass        48842 non-null  str  
 2   fnlwgt           48842 non-null  int64
 3   education        48842 non-null  str  
 4   educational-num  48842 non-null  int64
 5   marital-status   48842 non-null  str  
 6   occupation       48842 non-null  str  
 7   relationship     48842 non-null  str  
 8   race             48842 non-null  str  
 9   gender           48842 non-null  str  
 10  capital-gain     48842 non-null  int64
 11  capital-loss     48842 non-null  int64
 12  hours-per-week   48842 non-null  int64
 13  native-country   48842 non-null  str  
 14  income           48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


None

,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [5]:
# 將 ? 視為缺漏值
adult = df.replace('?', np.nan)

# 建立二元分類目標：>50K 為 1，<=50K 為 0
adult['income_binary'] = (adult['income'] == '>50K').astype(int)

adult['income_binary'].value_counts()

income_binary
0    37155
1    11687
Name: count, dtype: int64

In [6]:
# 觀察目標欄位與重要特徵的關係
pd.crosstab(adult['education'], adult['income'], normalize='index').sort_values('>50K', ascending=False).head(10)

income,<=50K,>50K
education,,
Prof-school,0.260192,0.739808
Doctorate,0.274411,0.725589
Masters,0.450884,0.549116
Bachelors,0.587165,0.412835
Assoc-acdm,0.742036,0.257964
Assoc-voc,0.746725,0.253275
Some-college,0.810351,0.189649
HS-grad,0.841422,0.158578
12th,0.926941,0.073059


In [7]:
# 取得數值型特徵、類別型特徵與目標變數
X_num = adult[['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']]

cat_cols = [
    'workclass',
    'education',
    'marital-status',
    'occupation',
    'relationship',
    'race',
    'gender',
    'native-country'
]

# 類別欄位的缺漏值使用眾數補上
for col in cat_cols:
    adult[col] = adult[col].fillna(adult[col].mode()[0])

X_cat = pd.get_dummies(adult[cat_cols])
y = adult['income_binary']

In [8]:
X_cat.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 99 columns):
 #   Column                                     Non-Null Count  Dtype
---  ------                                     --------------  -----
 0   workclass_Federal-gov                      48842 non-null  bool 
 1   workclass_Local-gov                        48842 non-null  bool 
 2   workclass_Never-worked                     48842 non-null  bool 
 3   workclass_Private                          48842 non-null  bool 
 4   workclass_Self-emp-inc                     48842 non-null  bool 
 5   workclass_Self-emp-not-inc                 48842 non-null  bool 
 6   workclass_State-gov                        48842 non-null  bool 
 7   workclass_Without-pay                      48842 non-null  bool 
 8   education_10th                             48842 non-null  bool 
 9   education_11th                             48842 non-null  bool 
 10  education_12th                             48842 non-null

In [9]:
# 切分訓練資料與測試資料
X_num_train, X_num_test, X_cat_train, X_cat_test, y_train, y_test = train_test_split(
    X_num, X_cat, y, test_size=0.2, random_state=0, stratify=y # stratify 參數確保訓練集與測試集的類別分布相似
)

# 數值型特徵標準化
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

# 合併數值型特徵與類別型特徵
X_train = np.concatenate([X_num_train_scaled, X_cat_train], axis=1)
X_test = np.concatenate([X_num_test_scaled, X_cat_test], axis=1)

In [10]:
# 選擇模型
logistic_model = LogisticRegression(
    solver='liblinear', 
    max_iter=1000, 
    random_state=0
)

# 定義超參數網格
param_grid = {
    'C': [0.01, 0.1, 1, 10], # 正則化強度的倒數，值越小表示正則化越強
    'penalty': ['l1', 'l2'] # 正則化類型，l1 表示 Lasso，l2 表示 Ridge，需要 solver='liblinear' 才能使用 l1
}

# 使用 GridSearchCV 進行超參數調整
grid_search = GridSearchCV(
    estimator=logistic_model,
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    n_jobs=2,
    verbose=1 # 1: 顯示簡單的進度訊息，2: 顯示詳細的進度訊息，0: 不顯示進度訊息
)

grid_search.fit(X_train, y_train)

print('Best Hyperparameters:', grid_search.best_params_)
print('Best CV F1-score:', grid_search.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best Hyperparameters: {'C': 10, 'penalty': 'l1'}
Best CV F1-score: 0.6555665681327297


In [11]:
# 使用 CV 後的最佳模型進行預測
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# 計算 F1-score 與混淆矩陣
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print('Test F1-score:', f1)
print('Confusion Matrix:')
cm_df = pd.DataFrame(
    cm,
    index=['Truth_Negative (0)', 'Truth_Positive (1)'],
    columns=['Pred_Negative (0)', 'Pred_Positive (1)']
)
print(cm_df)

Test F1-score: 0.6582159624413145
Confusion Matrix:
                    Pred_Negative (0)  Pred_Positive (1)
Truth_Negative (0)               6911                520
Truth_Positive (1)                936               1402


In [12]:
# 儲存 CV 後的最佳模型，以及預測時會用到的前處理物件
joblib.dump(best_model, './adult_income_logistic_model.pkl')
joblib.dump(scaler, './adult_income_scaler.pkl')
joblib.dump(X_cat.columns, './adult_income_dummy_columns.pkl')

['./adult_income_dummy_columns.pkl']

In [13]:
# 載入模型與前處理物件
model = joblib.load('./adult_income_logistic_model.pkl')
scaler = joblib.load('./adult_income_scaler.pkl')
dummy_columns = joblib.load('./adult_income_dummy_columns.pkl')

In [14]:
# 模擬使用者填寫一筆資料
# 數值欄位順序要跟 X_num 完全一樣
user_num = [[
    39,  # age
    13,  # educational-num
    0,   # capital-gain
    0,   # capital-loss
    40   # hours-per-week
]]

# 使用者填寫的類別欄位
user_workclass = 'Private'
user_education = 'Bachelors'
user_marital_status = 'Never-married'
user_occupation = 'Prof-specialty'
user_relationship = 'Not-in-family'
user_race = 'White'
user_gender = 'Male'
user_native_country = 'United-States'

# 數值型特徵標準化
user_num_scaled = scaler.transform(user_num)

# 類別型特徵 dummy
user_cat = [[0] * len(dummy_columns)]

user_cat_values = {
    'workclass': user_workclass,
    'education': user_education,
    'marital-status': user_marital_status,
    'occupation': user_occupation,
    'relationship': user_relationship,
    'race': user_race,
    'gender': user_gender,
    'native-country': user_native_country
}

# 根據使用者填寫的類別特徵值，將對應的 dummy 變數設為 1
for col, value in user_cat_values.items():
    dummy_col = col + '_' + value
    if dummy_col in dummy_columns:
        user_cat[0][dummy_columns.get_loc(dummy_col)] = 1

# 合併特徵
user_X = np.concatenate([user_num_scaled, user_cat], axis=1)

# 預測該使用者的收入類別
# 0 代表 <=50K，1 代表 >50K
pred_income = model.predict(user_X)

# 取得機率，pred_prob[0][0] 是 <=50K 的機率，pred_prob[0][1] 是 >50K 的機率
pred_prob = model.predict_proba(user_X)


print('預測結果:')
if pred_income[0] == 1:
    print('>50K')
else:
    print('<=50K')

print('-' * 30)

print('<=50K 機率:', pred_prob[0][0])
print('>50K 機率:', pred_prob[0][1])

預測結果:
<=50K
------------------------------
<=50K 機率: 0.8802349397994625
>50K 機率: 0.11976506020053748


延伸議題：
- 是不是所有特徵都對預測有幫助？可以透過特徵重要性分析來了解哪些特徵對模型預測影響較大，並考慮是否要保留或移除某些特徵。
- 如果要提升模型的效能，還有哪些方法可以嘗試？例如特徵工程、模型選擇、超參數調整等。
- 如果要將房價預測模型部署到線上系統，該怎麼做？

你可以用 AI 來協助你完成這些延伸議題的研究與實作，並且可以透過 AI 來幫助你分析特徵的重要性、選擇合適的模型、調整超參數，甚至將模型部署到線上系統。